## Interacting with Gemini LLM using Langchain

First, we need to install the `langchain-google-genai` and `google-generativeai` packages to enable interaction with Gemini models through Langchain.

In [ ]:
pip install -U langchain-google-genai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 1.6 MB/s eta 0:00:00


Next, we'll import the necessary libraries and set up your Google API Key. You'll need an API key from Google AI Studio. Please store it in Colab's secrets manager under the name `GOOGLE_API_KEY`.

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

# Set the API key from Colab secrets
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

# Initialize the Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Invoke the model with a simple prompt
response = llm.invoke("What is the capital of France?")

# Print the response
print(response.content)


The capital of France is **Paris**.


Now, we can initialize the Gemini model using Langchain's `ChatGoogleGenerativeAI` and make a simple request.

In [ ]:
# Example interaction
response = llm.invoke("Write a short, catchy slogan for a coffee shop that specializes in unique coffee blends.")
print(response.content)


Here are a few options, playing with different angles:

**Short & Punchy:**

*   **Blends with Character.**
*   **Sip Beyond the Ordinary.**
*   **Your Next Favorite Blend Awaits.**
*   **Distinctly Brewed. Uniquely Yours.**

**A Bit More Descriptive:**

*   **Unique Blends, Unforgettable Sips.**
*   **The Art of the Unique Blend.**
*   **Crafting Your Perfect, Peculiar Cup.**

**My Top Picks:**

1.  **Sip Beyond the Ordinary.** (Intriguing, promises something new)
2.  **Unique Blends, Unforgettable Sips.** (Direct, highlights benefit and specialty)
3.  **Your Next Favorite Blend Awaits.** (Creates anticipation and curiosity)


# Langchain Ingestion and Retrieval Workflow
We will now set up a Retrieval-Augmented Generation (RAG) workflow. This involves ingesting a document into a vector store and then using an LLM to answer questions based on the retrieved information.



In [ ]:
# Install additional necessary packages
%pip install -U chromadb tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

# Sample HR Policy Document

In [ ]:
# Install necessary packages for document loading if not already installed
%pip install -U langchain-community

# Define a dummy HR Policy Document
hr_policy_text = """# HR Policy Handbook

## Section 1: Employee Conduct

Employees are expected to conduct themselves professionally and ethically at all times. This includes respecting colleagues, clients, and company property. Harassment of any kind is strictly prohibited and will lead to disciplinary action, up to and including termination.

## Section 2: Leave Policy

### 2.1 Annual Leave

All full-time employees are entitled to 20 days of annual leave per year. Leave requests must be submitted at least two weeks in advance through the HR portal and approved by a direct manager. Unused leave days do not roll over to the next year and must be utilized within the current calendar year.

### 2.2 Sick Leave

Employees are allocated 10 days of paid sick leave per year. For absences exceeding three consecutive days, a doctor's note is required. All sick leave must be reported to the manager and HR within 24 hours of absence.

## Section 3: Expense Reimbursement

Employees can claim reimbursement for business-related expenses. Original receipts must be attached to all expense reports. Expenses must be submitted within 30 days of incurring them. The company will not reimburse personal expenses or expenses without valid receipts.

## Section 4: Remote Work Policy

Remote work arrangements are subject to manager approval and depend on job role suitability. Employees working remotely must maintain a secure and productive work environment. The company provides a stipend for essential home office equipment, upon request and approval.

## Section 5: Performance Reviews

Performance reviews are conducted semi-annually, in June and December, to assess employee performance and provide feedback for growth and development. Employees will have the opportunity to discuss their performance with their managers and set future goals."""

# Load the document using Langchain's TextLoader
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

# Create a Document object directly from the string
documents = [Document(page_content=hr_policy_text, metadata={"source": "HR Policy Handbook"})]

print(f"Loaded {len(documents)} document.")
print("First 200 characters of the document:")
print(documents[0].page_content[:200])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.3.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.43.0 which is incompatible.
google-adk 2.3.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.43.0 which is incompatible.


/tmp/ipykernel_11816/733666046.py:34: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Loaded 1 document.
First 200 characters of the document:
# HR Policy Handbook

## Section 1: Employee Conduct

Employees are expected to conduct themselves professionally and ethically at all times. This includes respecting colleagues, clients, and company 


# Document into Chunks

In [ ]:
%pip install -U langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, # Max size of each chunk
    chunk_overlap=100, # Overlap between chunks to maintain context
    length_function=len,
    add_start_index=True,
)

# Split the document into chunks
chunks = text_splitter.split_documents(documents)

print(f"Split document into {len(chunks)} chunks.")
print("First chunk content:")
print(chunks[0].page_content)

Split document into 6 chunks.
First chunk content:
# HR Policy Handbook

## Section 1: Employee Conduct

Employees are expected to conduct themselves professionally and ethically at all times. This includes respecting colleagues, clients, and company property. Harassment of any kind is strictly prohibited and will lead to disciplinary action, up to and including termination.

## Section 2: Leave Policy

### 2.1 Annual Leave


Initialize Google Generative AI Embeddings

In [ ]:
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize the embedding model
# Make sure GOOGLE_API_KEY is available from previous cells
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=GOOGLE_API_KEY)

print("Google Generative AI Embeddings initialized.")

Google Generative AI Embeddings initialized.


# Chroma Vector Store

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata

# Get the API key (ensure it's available for this cell's execution)
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# Re-initialize the embedding model with 'models/embedding-001'
# Both 'models/embedding-001' and 'text-embedding-004' are failing with NOT_FOUND errors.
# Reverting to 'models/embedding-001' as it was the model that initialized successfully in a prior cell.
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2", google_api_key=GOOGLE_API_KEY)

# Create a Chroma vector store from the document chunks and embeddings
# We'll create an in-memory Chroma instance for this demonstration.
vectorstore = Chroma.from_documents(chunks, embeddings)

print(f"Chroma vector store created with {vectorstore._collection.count()} entries.")

# Create a retriever from the vector store
retriever = vectorstore.as_retriever()

print("Retriever created from Chroma vector store.")

Chroma vector store created with 6 entries.
Retriever created from Chroma vector store.


# Retrieval-Augmented Generation (RAG) Chain

In [ ]:
# The original %pip install command is removed to avoid dependency conflicts and is not needed for a functional RAG setup.
# The original imports for chain functions are removed as per the request to write code without chains.
# from langchain_classic.chains.combine_documents import create_stuff_documents_chain
# from langchain_classic.chains import create_retrieval_chain

from langchain_core.prompts import ChatPromptTemplate

# Define the prompt template for our LLM
# The context will be filled by the retrieved documents
prompt = ChatPromptTemplate.from_template(
    """Answer the user's question based on the provided context.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.

    Context: {context}
    Question: {input}"""
)

# Define a custom class to simulate the behavior of the retrieval_chain
# without using Langchain's chain objects.
class CustomRetrievalChain:
    def invoke(self, input_dict):
        # Assumes 'llm' and 'retriever' are available in the global scope
        # from previously executed cells.
        question = input_dict["input"]

        # 1. Retrieve relevant documents using the retriever
        retrieved_docs = retriever.invoke(question)

        # 2. Format the retrieved documents into a single context string
        #    Each document's page_content is joined by double newlines.
        formatted_context = "\n\n".join([doc.page_content for doc in retrieved_docs])

        # 3. Format the prompt with the context and the user's question
        #    ChatPromptTemplate.format_messages returns a list of Message objects.
        formatted_prompt_messages = prompt.format_messages(
            context=formatted_context,
            input=question
        )

        # 4. Invoke the LLM with the formatted prompt messages
        response = llm.invoke(formatted_prompt_messages)

        # The result should match the structure of the original retrieval_chain's output.
        return {"answer": response.content}

# Instantiate the custom retrieval chain, so subsequent cells can call retrieval_chain.invoke()
retrieval_chain = CustomRetrievalChain()

print("RAG 'retrieval_chain' (manual implementation) created successfully.")

RAG 'retrieval_chain' (manual implementation) created successfully.


# Retrieval

In [ ]:
# Ask a question about the HR policy
question = "How many sick leave are available?"
print(f"\nQuestion: {question}")

# Invoke the RAG chain
response = retrieval_chain.invoke({"input": question})
print(f"\nAnswer: {response['answer']}")

question_2 = "Can I take a leave without pay?"
print(f"\nQuestion: {question_2}")
response_2 = retrieval_chain.invoke({"input": question_2})
print(f"\nAnswer: {response_2['answer']}")

question_3 = "How many PTO days are avaible for employees?"
print(f"\nQuestion: {question_3}")
response_3 = retrieval_chain.invoke({"input": question_3})
print(f"\nAnswer: {response_3['answer']}")


Question: How many sick leave are available?

Answer: Employees are allocated 10 days of paid sick leave per year.

Question: Can I take a leave without pay?

Answer: I don't know the answer based on the provided context. The document outlines policies for annual leave and sick leave, both of which are paid, but does not mention leave without pay.

Question: How many PTO days are avaible for employees?

Answer: Employees are entitled to 20 days of annual leave and 10 days of paid sick leave per year.
